In [111]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
import joblib
import os

In [112]:
landmark_path = "D:\\IDE Repo\\Dl-net\\data\\casme2-preprocessed-v2\\face_landmarks.csv"   
train_path = "D:\\IDE Repo\\Dl-net\\data\\casme2-preprocessed-v2\\train.csv" 
SAVE_DIR = "processed"
os.makedirs(SAVE_DIR, exist_ok=True)

In [113]:
landmarks_df = pd.read_csv(landmark_path)
train_df = pd.read_csv(train_path)
print("=" * 60)
print("Landmarks Dataset")
print(landmarks_df.shape)
print("=" * 60)
print("Train Dataset")
print(train_df.shape)
landmarks_df.head()


Landmarks Dataset
(500, 1436)
Train Dataset
(13615, 14)


,filepath,landmarks_detected,x_0,y_0,z_0,x_1,y_1,z_1,x_2,y_2,...,z_474,x_475,y_475,z_475,x_476,y_476,z_476,x_477,y_477,z_477
0,./data/casme2-preprocessed-v2\CASME2 Preproces...,478,0.514768,0.722219,-0.136648,0.523118,0.576818,-0.294961,0.517824,0.617790,...,0.025961,0.768488,0.236143,0.025960,0.713368,0.271990,0.025894,0.768613,0.307834,0.025907
1,./data/casme2-preprocessed-v2\CASME2 Preproces...,478,0.503075,0.703677,-0.148351,0.493394,0.541940,-0.285421,0.497619,0.590272,...,0.017879,0.768665,0.234639,0.017880,0.715009,0.270061,0.017815,0.770119,0.305565,0.017829
2,./data/casme2-preprocessed-v2\CASME2 Preproces...,478,0.500352,0.686904,-0.110848,0.499011,0.567026,-0.270116,0.498298,0.594931,...,0.000029,0.764490,0.239815,0.000024,0.711010,0.275667,-0.000036,0.763916,0.310359,-0.000028
3,./data/casme2-preprocessed-v2\CASME2 Preproces...,478,0.501748,0.691448,-0.144714,0.504308,0.553497,-0.295761,0.501742,0.600460,...,0.034370,0.796956,0.252263,0.034371,0.740294,0.285858,0.034301,0.794857,0.321623,0.034317
4,./data/casme2-preprocessed-v2\CASME2 Preproces...,478,0.485267,0.679308,-0.135492,0.496794,0.537978,-0.274870,0.493410,0.576039,...,0.013004,0.766617,0.254572,0.013000,0.715819,0.286032,0.012945,0.764681,0.318232,0.012957


In [114]:
train_df.head()

,filepath,extension,width,height,resolution,mode,aspect_ratio,brightness,contrast,sharpness,edge_density,entropy,noise,class
0,./data/casme2-preprocessed-v2/CASME2 Preproces...,.jpg,259,308,79772,RGB,0.841,90.150930,36.217449,59.012542,0.010492,7.070181,92.900850,surprise
1,./data/casme2-preprocessed-v2/CASME2 Preproces...,.jpg,259,313,81067,RGB,0.827,113.415163,37.030567,34.598467,0.005847,7.131897,100.353917,disgust
2,./data/casme2-preprocessed-v2/CASME2 Preproces...,.jpg,263,315,82845,RGB,0.835,84.742664,29.501137,28.869772,0.008896,6.793766,88.410561,disgust
3,./data/casme2-preprocessed-v2/CASME2 Preproces...,.jpg,284,321,91164,RGB,0.885,95.845882,40.332515,39.903049,0.010969,6.938547,93.265230,happy
4,./data/casme2-preprocessed-v2/CASME2 Preproces...,.jpg,256,312,79872,RGB,0.821,114.856971,36.516864,44.872642,0.008075,7.082653,103.252668,disgust


In [115]:
print('landmarks filepath sample:')
print(landmarks_df['filepath'].head(10).to_string(index=False))
print('\ntrain filepath sample:')
print(train_df['filepath'].head(10).to_string(index=False))
print('\nlandmarks basename sample:')
print(landmarks_df['filepath'].astype(str).str.replace('\\\\', '/', regex=False).str.split('/').str[-1].head(10).to_string(index=False))
print('\ntrain basename sample:')
print(train_df['filepath'].astype(str).str.replace('\\\\', '/', regex=False).str.split('/').str[-1].head(10).to_string(index=False))

landmarks filepath sample:
./data/casme2-preprocessed-v2\CASME2 Preprocess...
./data/casme2-preprocessed-v2\CASME2 Preprocess...
./data/casme2-preprocessed-v2\CASME2 Preprocess...
./data/casme2-preprocessed-v2\CASME2 Preprocess...
./data/casme2-preprocessed-v2\CASME2 Preprocess...
./data/casme2-preprocessed-v2\CASME2 Preprocess...
./data/casme2-preprocessed-v2\CASME2 Preprocess...
./data/casme2-preprocessed-v2\CASME2 Preprocess...
./data/casme2-preprocessed-v2\CASME2 Preprocess...
./data/casme2-preprocessed-v2\CASME2 Preprocess...

train filepath sample:
./data/casme2-preprocessed-v2/CASME2 Preprocess...
./data/casme2-preprocessed-v2/CASME2 Preprocess...
./data/casme2-preprocessed-v2/CASME2 Preprocess...
./data/casme2-preprocessed-v2/CASME2 Preprocess...
./data/casme2-preprocessed-v2/CASME2 Preprocess...
./data/casme2-preprocessed-v2/CASME2 Preprocess...
./data/casme2-preprocessed-v2/CASME2 Preprocess...
./data/casme2-preprocessed-v2/CASME2 Preprocess...
./data/casme2-preprocessed-v2/C

In [116]:
# Data from train.csv
label_column = "class"

# Landmark columns from face_landmarks.csv
x_cols = sorted(
    [c for c in landmarks_df.columns if c.startswith("x_")],
    key=lambda x: int(x.split("_")[1]),
)
y_cols = sorted(
    [c for c in landmarks_df.columns if c.startswith("y_")],
    key=lambda x: int(x.split("_")[1]),
)
z_cols = sorted(
    [c for c in landmarks_df.columns if c.startswith("z_")],
    key=lambda x: int(x.split("_")[1]),
)

print("Number of X landmarks:", len(x_cols))
print("Number of Y landmarks:", len(y_cols))
print("Number of Z landmarks:", len(z_cols))

assert len(x_cols) == len(y_cols) == len(z_cols), "Landmark columns do not match."
num_landmarks = len(x_cols)

print("Detected Landmarks:", num_landmarks)

Number of X landmarks: 478
Number of Y landmarks: 478
Number of Z landmarks: 478
Detected Landmarks: 478


In [117]:
# ----------------------------------------------------------
# Create Feature Matrix from multiple datasets (CASME2, CK+, SAMM)
# Uses x,y coordinates only to match model in_channels=2
# ----------------------------------------------------------

datasets = [
    {
        "name": "casme2",
        "landmark": "D:\\IDE Repo\\Dl-net\\data\\casme2-preprocessed-v2\\face_landmarks.csv",
        "train": "D:\\IDE Repo\\Dl-net\\data\\casme2-preprocessed-v2\\train.csv",
    },
    {
        "name": "ckplus",
        "landmark": "D:\\IDE Repo\\Dl-net\\data\\ckplusferdata\\face_landmarks.csv",
        "train": "D:\\IDE Repo\\Dl-net\\data\\ckplusferdata\\train.csv",
    },
    {
        "name": "samm",
        "landmark": "D:\\IDE Repo\\Dl-net\\data\\sammassamexpression\\face_landmarks.csv",
        "train": "D:\\IDE Repo\\Dl-net\\data\\sammassamexpression\\train.csv",
    },
]

merged_list = []
label_column = "class"

for ds in datasets:
    if not (os.path.exists(ds["landmark"]) and os.path.exists(ds["train"])):
        print(f"Skipping {ds['name']} — files not found: {ds['landmark']} or {ds['train']}")
        continue

    ldf = pd.read_csv(ds["landmark"]).copy()
    tdf = pd.read_csv(ds["train"]).copy()

    # normalize paths and merge
    ldf["filepath_norm"] = ldf["filepath"].astype(str).str.strip().str.replace("\\", "/", regex=False)
    tdf["filepath_norm"] = tdf["filepath"].astype(str).str.strip().str.replace("\\", "/", regex=False)
    
    print(f"{ds['name']} - Sample landmark paths:", ldf["filepath_norm"].head(2).tolist())
    print(f"{ds['name']} - Sample train paths:", tdf["filepath_norm"].head(2).tolist())

    # determine x/y cols in this landmark file
    x_cols_local = sorted([c for c in ldf.columns if c.startswith("x_")], key=lambda x: int(x.split("_")[1]))
    y_cols_local = sorted([c for c in ldf.columns if c.startswith("y_")], key=lambda x: int(x.split("_")[1]))

    if len(x_cols_local) == 0 or len(y_cols_local) == 0:
        print(f"No x/y landmarks found for {ds['name']}, skipping")
        continue

    # select only x and y columns
    use_cols = ["filepath_norm"] + x_cols_local + y_cols_local
    ldf_sel = ldf[use_cols]

    merged = ldf_sel.merge(tdf[["filepath_norm", label_column]], on="filepath_norm", how="inner")
    print(f"{ds['name']} merged:", merged.shape)

    if merged.empty:
        print(f"Warning: no rows matched for {ds['name']}")
        continue

    merged_list.append(merged)

if not merged_list:
    raise RuntimeError("No dataset merged — check that data files exist and filepath columns match.")

# concatenate merged datasets
merged_all = pd.concat(merged_list, ignore_index=True)
print("Combined merged dataset:", merged_all.shape)

# Determine common landmark indices (assumes same numbering scheme)
x_cols = sorted([c for c in merged_all.columns if c.startswith("x_")], key=lambda x: int(x.split("_")[1]))
y_cols = sorted([c for c in merged_all.columns if c.startswith("y_")], key=lambda x: int(x.split("_")[1]))

# Keep the minimum length in case one dataset had fewer landmarks
min_len = min(len(x_cols), len(y_cols))
if min_len == 0:
    raise RuntimeError("No x/y landmark columns found in combined data.")

x_cols = x_cols[:min_len]
y_cols = y_cols[:min_len]

print("Using landmarks count:", min_len)

features = []
for i in range(min_len):
    features.append(x_cols[i])
    features.append(y_cols[i])

# Build X and y
X = merged_all[features]
encoder = LabelEncoder()
y = encoder.fit_transform(merged_all[label_column].astype(str).values.ravel())

print("Feature Matrix:", X.shape)
print("Label Vector:", y.shape)

print("Classes:")
for i, c in enumerate(encoder.classes_):
    print(i, "->", c)

# Handle missing values
imputer = SimpleImputer(strategy="median")
X = pd.DataFrame(imputer.fit_transform(X), columns=features)

# Save preprocessing artifacts
os.makedirs(SAVE_DIR, exist_ok=True)
joblib.dump(encoder, os.path.join(SAVE_DIR, "label_encoder.joblib"))
joblib.dump(imputer, os.path.join(SAVE_DIR, "imputer.joblib"))
joblib.dump(features, os.path.join(SAVE_DIR, "feature_columns.joblib"))

print("Saved preprocessing artifacts to:", SAVE_DIR)

casme2 - Sample landmark paths: ['./data/casme2-preprocessed-v2/CASME2 Preprocessed v2/others/reg_img65 (24).jpg', './data/casme2-preprocessed-v2/CASME2 Preprocessed v2/others/reg_img135 (14).jpg']
casme2 - Sample train paths: ['./data/casme2-preprocessed-v2/CASME2 Preprocessed v2/surprise/reg_img44 (4).jpg', './data/casme2-preprocessed-v2/CASME2 Preprocessed v2/disgust/reg_img70 (18).jpg']
casme2 merged: (393, 958)
ckplus - Sample landmark paths: ['./data/ckplusferdata/train/neutral/Training_46272799.jpg', './data/ckplusferdata/train/sad/Training_8856969.jpg']
ckplus - Sample train paths: ['./data/ckplusferdata/train/angry/Training_23412616.jpg', './data/ckplusferdata/train/fear/Training_9013309.jpg']
ckplus merged: (373, 958)
samm - Sample landmark paths: ['./data/sammassamexpression/expression_data/sad/image0020786_face.jpg', './data/sammassamexpression/expression_data/neutral/ffhq_2900_face.jpg']
samm - Sample train paths: ['./data/sammassamexpression/expression_data/happy/ffhq_401

In [118]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Train set:", X_train.shape, y_train.shape)
print("Test set:", X_test.shape, y_test.shape)


Train set: (928, 956) (928,)
Test set: (233, 956) (233,)


In [119]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.nn import (
    GCNConv,
    ChebConv,
    TopKPooling,
    global_mean_pool
)


# -------------------------------------------------------
# Spatial Branch (SPAGNN)
# -------------------------------------------------------
class SPAGNN(nn.Module):
    def __init__(self, in_channels, hidden_channels=64):
        super().__init__()

        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.pool1 = TopKPooling(hidden_channels, ratio=0.8)

        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.pool2 = TopKPooling(hidden_channels, ratio=0.8)

        self.fc = nn.Linear(hidden_channels, hidden_channels)

    def forward(self, data):

        x = data.x
        edge_index = data.edge_index
        batch = data.batch

        x = F.relu(self.conv1(x, edge_index))

        x, edge_index, _, batch, _, _ = self.pool1(
            x,
            edge_index,
            batch=batch
        )

        x = F.relu(self.conv2(x, edge_index))

        x, edge_index, _, batch, _, _ = self.pool2(
            x,
            edge_index,
            batch=batch
        )

        x = global_mean_pool(x, batch)

        x = self.fc(x)

        return x


# -------------------------------------------------------
# Spectral Branch (SPEGNN)
# -------------------------------------------------------
class SPEGNN(nn.Module):

    def __init__(self,
                 in_channels,
                 hidden_channels=64,
                 K=3):
        super().__init__()

        self.conv = ChebConv(
            in_channels,
            hidden_channels,
            K=K
        )

        self.fc = nn.Linear(hidden_channels,
                            hidden_channels)

    def forward(self, data):

        x = F.relu(
            self.conv(data.x,
                      data.edge_index)
        )

        x = global_mean_pool(
            x,
            data.batch
        )

        x = self.fc(x)

        return x


# -------------------------------------------------------
# Complete SSGNN
# -------------------------------------------------------
class SSGNN(nn.Module):

    def __init__(self,
                 in_channels,
                 hidden_channels,
                 num_classes):

        super().__init__()

        self.spatial = SPAGNN(
            in_channels,
            hidden_channels
        )

        self.spectral = SPEGNN(
            in_channels,
            hidden_channels
        )

        self.dropout = nn.Dropout(0.5)

        self.classifier = nn.Sequential(

            nn.Linear(hidden_channels * 2,
                      hidden_channels),

            nn.ReLU(),

            nn.Dropout(0.5),

            nn.Linear(hidden_channels,
                      num_classes)
        )

    def forward(self, data):

        spatial_feat = self.spatial(data)

        spectral_feat = self.spectral(data)

        feature = torch.cat(
            [spatial_feat, spectral_feat],
            dim=1
        )

        feature = self.dropout(feature)

        out = self.classifier(feature)

        return out
    
print("\n--- 2. SSGNN Model ---")

model = SSGNN(
    in_channels=2,      # (x,y landmark coordinates)
    hidden_channels=128,
    num_classes=7      # CASME II classes
)
print(model)


--- 2. SSGNN Model ---
SSGNN(
  (spatial): SPAGNN(
    (conv1): GCNConv(2, 128)
    (pool1): TopKPooling(128, ratio=0.8, multiplier=1.0)
    (conv2): GCNConv(128, 128)
    (pool2): TopKPooling(128, ratio=0.8, multiplier=1.0)
    (fc): Linear(in_features=128, out_features=128, bias=True)
  )
  (spectral): SPEGNN(
    (conv): ChebConv(2, 128, K=3, normalization=sym)
    (fc): Linear(in_features=128, out_features=128, bias=True)
  )
  (dropout): Dropout(p=0.5, inplace=False)
  (classifier): Sequential(
    (0): Linear(in_features=256, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.5, inplace=False)
    (3): Linear(in_features=128, out_features=7, bias=True)
  )
)


In [120]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(device)

cuda


In [121]:
from sklearn.utils.class_weight import compute_class_weight
weights = compute_class_weight('balanced', classes=np.unique(y), y=y)
criterion = torch.nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float).to(device))


optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=5e-4
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=7)

In [122]:
from tqdm import tqdm

def train(model, loader, optimizer, criterion):
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for data in tqdm(loader):

        data = data.to(device)

        optimizer.zero_grad()

        out = model(data)

        loss = criterion(out, data.y)

        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        total_loss += loss.item()

        pred = out.argmax(dim=1)

        correct += (pred == data.y).sum().item()

        total += data.y.size(0)

    loss = total_loss / len(loader)
    acc = correct / total

    return loss, acc

In [123]:
@torch.no_grad()
def evaluate(model, loader, criterion):

    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    for data in loader:

        data = data.to(device)

        out = model(data)

        loss = criterion(out, data.y)

        total_loss += loss.item()

        pred = out.argmax(dim=1)

        correct += (pred == data.y).sum().item()

        total += data.y.size(0)

    loss = total_loss / len(loader)
    acc = correct / total

    return loss, acc

In [124]:
import torch
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

edges = []

for i in range(67):
    edges.append([i, i + 1])
    edges.append([i + 1, i])

edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

print(edge_index.shape)


def create_graph_dataset(X, y, edge_index, in_channels=2):

    # Accept pandas DataFrame or numpy array
    if hasattr(X, "values"):
        X_np = X.values
    else:
        X_np = np.asarray(X)

    dataset = []

    num_features = X_np.shape[1]
    if num_features % in_channels != 0:
        raise ValueError(f"Number of features ({num_features}) is not divisible by in_channels={in_channels}.")

    num_nodes = num_features // in_channels

    for i in range(len(X_np)):

        # reshape from (num_features,) -> (num_nodes, in_channels)
        row = X_np[i]
        x = torch.tensor(row.reshape(num_nodes, in_channels), dtype=torch.float)

        label = torch.tensor(
            int(y[i]),
            dtype=torch.long,
        )

        graph = Data(
            x=x,
            edge_index=edge_index,
            y=label
        )

        dataset.append(graph)

    return dataset

train_dataset = create_graph_dataset(
    X_train,
    y_train,
    edge_index,
    in_channels=2,
)

# If your features include z-coordinates, remove them or set in_channels=3 and adjust the model.
test_dataset = create_graph_dataset(
    X_test,
    y_test,
    edge_index,
    in_channels=2,
)

# Create PyG DataLoaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("Datasets -> train:, test:", len(train_dataset), len(test_dataset))
print("DataLoaders -> train batches:, test batches:", len(train_loader), len(test_loader))

torch.Size([2, 134])
Datasets -> train:, test: 928 233
DataLoaders -> train batches:, test batches: 29 8


In [154]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape


((928, 956), (928,), (233, 956), (233,))

In [125]:
EPOCHS = 100

best_acc = 0

train_losses = []
test_losses = []

train_accs = []
test_accs = []

for epoch in range(1, EPOCHS + 1):

    train_loss, train_acc = train(
        model,
        train_loader,
        optimizer,
        criterion
    )

    test_loss, test_acc = evaluate(
        model,
        test_loader,
        criterion
    )
    
    scheduler.step(test_loss)
    
    current_lr = optimizer.param_groups[0]['lr']
    

    train_losses.append(train_loss)
    test_losses.append(test_loss)

    train_accs.append(train_acc)
    test_accs.append(test_acc)

    if test_acc > best_acc:

        best_acc = test_acc

        torch.save(model.state_dict(), "best_ssgnn.pth")

    print(
        f"Epoch {epoch:03d} | "
        f"Train Loss={train_loss:.4f} Train Acc={train_acc:.4f} | "
        f"Test Loss={test_loss:.4f} Test Acc={test_acc:.4f} | "
        f"LR={current_lr:.6f}"
    )

print("\nTraining Finished!")
print("Best Accuracy:", best_acc)

100%|██████████| 29/29 [00:00<00:00, 37.39it/s]


Epoch 001 | Train Loss=1.9502 Train Acc=0.1110 | Test Loss=1.9482 Test Acc=0.0815 | LR=0.001000


100%|██████████| 29/29 [00:00<00:00, 51.20it/s]


Epoch 002 | Train Loss=1.9474 Train Acc=0.1325 | Test Loss=1.9458 Test Acc=0.2189 | LR=0.001000


100%|██████████| 29/29 [00:00<00:00, 52.03it/s]


...

Epoch 098 | Train Loss=1.7704 Train Acc=0.1649 | Test Loss=1.7596 Test Acc=0.1459 | LR=0.000001


100%|██████████| 29/29 [00:00<00:00, 49.41it/s]


Epoch 099 | Train Loss=1.7834 Train Acc=0.1713 | Test Loss=1.7596 Test Acc=0.1459 | LR=0.000001


100%|██████████| 29/29 [00:00<00:00, 51.81it/s]


Epoch 100 | Train Loss=1.7758 Train Acc=0.1756 | Test Loss=1.7596 Test Acc=0.1459 | LR=0.000001

Training Finished!
Best Accuracy: 0.351931330472103
